# Partial-Label Masked Multi-Task Learning

Train the same shared encoder + eight sigmoid heads as step 03, but with masked `BCEWithLogitsLoss`: each emotion head is updated ONLY by sentences whose label was actually observed in the original benchmark. The unobserved cells (NA in Dataset C) contribute zero gradient.

Evaluation is similarly masked — per-emotion metrics use only test cells where the emotion was observed. The head-to-head comparison with step 03 in step 05 (bootstrap) is the central claim of the paper.

Wraps `scripts/mtl_partial_masked.py`. **GPU required.**

## 0. Install / check packages

In [ ]:
import importlib.util
for pkg in ('numpy', 'pandas', 'torch', 'transformers'):
    print(f"{pkg:14s}: {'OK' if importlib.util.find_spec(pkg) else 'MISSING'}")

## 1. Setup

In [ ]:
import os, sys, torch
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)
print('CUDA available:', torch.cuda.is_available())

## 2. Train and evaluate

In [ ]:
!python scripts/mtl_partial_masked.py \
    --model bert-base-uncased \
    --epochs 3 --batch-size 32 --max-length 128 --lr 2e-5 --seed 42

## 3. Per-emotion metrics (observed cells only)

In [ ]:
import pandas as pd
out = REPO_ROOT / '04_PartialLabelMaskedMTL' / 'outputs'
metrics = pd.read_csv(out / '04_mtl_masked_metrics.csv')
metrics.round(3)

## 4. Macro averages (skipping NaNs)

In [ ]:
metrics[['precision', 'recall', 'f1', 'auroc']].mean(skipna=True).round(3)